In [5]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import seaborn as sns
import scipy
import sklearn
from sklearn.svm import SVC
from scipy.stats import pearsonr
from sklearn import datasets, linear_model
from sklearn import preprocessing
from sklearn.pipeline import make_pipeline
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import (
    train_test_split, 
    StratifiedKFold, cross_val_score,  
    RepeatedStratifiedKFold, 
    RandomizedSearchCV,
    train_test_split, 
    KFold
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report, confusion_matrix
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor, plot_tree, DecisionTreeClassifier
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve,
    f1_score
)
from sklearn.feature_selection import SelectFromModel
from sklearn.compose import ColumnTransformer
from scipy.stats import loguniform
from scipy import sparse
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import make_scorer, balanced_accuracy_score, f1_score

import category_encoders as ce



In [1]:
artifact = joblib.load("svm_classification_model.joblib")
svm_model = artifact["model"]
best_thr = artifact["threshold"]

In [13]:
df = pd.read_excel("FinalTestDataset2025.xls")
id_field = df["ID"]
df = df.drop("ID", axis=1)

cat_cols = ['PgR', 'HER2', 'TrippleNegative', 'ChemoGrade',
       'Proliferation', 'HistologyType', 'LNStatus', 'Gene']

# cast all object type in train dataset to object 
for col in cat_cols:
    df[col] = df[col].astype("object")

# replace missing values with NA
df.replace(999, pd.NA, inplace=True)

# Iterative Imputation

df_imputed = df.copy()

## Select categorical columns
cat_cols_to_encode = df_imputed.select_dtypes(include=['object', 'category']).columns

encoder = ce.OrdinalEncoder(handle_missing='return_nan') 


## Encode categorical features
df_imputed[cat_cols_to_encode] = encoder.fit_transform(df_imputed[cat_cols_to_encode])


## Imputation with IterativeImputer
imputer = IterativeImputer(max_iter=20, random_state=42)
X_imputed = imputer.fit_transform(df_imputed)

# Convert back to DataFrame
X_imputed_df = pd.DataFrame(X_imputed,
                            columns=df_imputed.columns,
                            index=df_imputed.index)

# Round encoded categorical columns back to integers
for col in cat_cols_to_encode:
    max_val = df_imputed[col].max()
    min_val = df_imputed[col].min()
    X_imputed_df[col] = X_imputed_df[col].clip(lower=min_val, upper=max_val)
    X_imputed_df[col] = X_imputed_df[col].round().astype(int)

    
# Inverse transform encoded categorical columns
X_imputed_df[cat_cols_to_encode] = encoder.inverse_transform(X_imputed_df[cat_cols_to_encode])


# Recombine target
final_df = pd.concat([X_imputed_df], axis=1)
final_df = final_df[sorted(final_df.columns)]
print("\n--- Missing Value Check ---")
print(final_df.isna().sum().sum())


--- Missing Value Check ---
0


In [14]:
scores_new = svm_model.decision_function(final_df)
y_pred = (scores_new >= best_thr).astype(int)

In [15]:
predicted_df = pd.DataFrame({
    "ID": id_field.values,
    "pCR": y_pred,
    })
    

In [16]:
predicted_df.to_csv("PCRPrediction.csv", index=False)
